# 00 Load Library
---


In [ ]:
from pathlib import Path
import pandas as pd


# 01 Inisialisasi Path Direktori
---
## Konfigurasi Folder BPS
---
### Menentukan Lokasi File Mentah BPS dan Output
---
Menyiapkan path direktori data indikator makroekonomi BPS DIY.


In [ ]:
BASE_DIR = Path.cwd().parent.parent if Path.cwd().name == 'data_cleaning' else Path.cwd()
BPS_RAW_DIR = BASE_DIR / 'data' / 'raw' / 'bps'
OUTPUT_DIR = BASE_DIR / 'data' / 'cleaned'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# 02 Pembacaan Data PDRB per Kapita ADHB 2024
---
## Parsing Tabel PDRB
---
### Ekstraksi Nilai PDRB per Kabupaten Kota
---
Membaca data PDRB per kapita atas dasar harga berlaku tahun 2024 dari publikasi BPS DIY.


In [ ]:
pdrb_file = list(BPS_RAW_DIR.glob('*PDRB*'))[0]
df_pdrb_raw = pd.read_csv(pdrb_file, skiprows=3)
df_pdrb = df_pdrb_raw.dropna(how='all').copy()
df_pdrb.columns = ['wilayah_raw', 'pdrb_per_kapita_ribu_rp']
df_pdrb.head()


# 03 Pembacaan Rata-rata Pengeluaran per Kapita Bulanan
---
## Parsing Tabel Pengeluaran
---
### Ekstraksi Daya Beli Riil Konsumen
---
Membaca data rata-rata pengeluaran per kapita sebulan tahun 2024 dari BPS DIY.


In [ ]:
pengeluaran_file = list(BPS_RAW_DIR.glob('*Pengeluaran*'))[0]
df_pengeluaran_raw = pd.read_csv(pengeluaran_file, skiprows=3)
df_pengeluaran = df_pengeluaran_raw.dropna(how='all').copy()
df_pengeluaran.columns = ['wilayah_raw', 'pengeluaran_per_kapita_rp']
df_pengeluaran.head()


# 04 Penanganan Anomali Baris Agregat Provinsi
---
## Blacklist Filtering
---
### Eliminasi Baris Total D.I. Yogyakarta
---
Menghapus baris total provinsi agar tidak terhitung ganda sebagai entitas kabupaten ke-6.


In [ ]:
wilayah_map = {
    'kulon progo': 'kulon_progo',
    'bantul': 'bantul',
    'gunungkidul': 'gunungkidul',
    'gunung kidul': 'gunungkidul',
    'sleman': 'sleman',
    'kota yogyakarta': 'kota_yogyakarta',
    'yogyakarta': 'kota_yogyakarta'
}
def clean_wilayah(s):
    s_clean = str(s).strip().lower()
    if 'd.i. yogyakarta' in s_clean or 'daerah istimewa' in s_clean:
        return None
    for k, v in wilayah_map.items():
        if k in s_clean:
            return v
    return None


In [ ]:
df_pdrb['wilayah'] = df_pdrb['wilayah_raw'].apply(clean_wilayah)
df_pdrb_clean = df_pdrb.dropna(subset=['wilayah'])[['wilayah', 'pdrb_per_kapita_ribu_rp']].copy()
df_pengeluaran['wilayah'] = df_pengeluaran['wilayah_raw'].apply(clean_wilayah)
df_pengeluaran_clean = df_pengeluaran.dropna(subset=['wilayah'])[['wilayah', 'pengeluaran_per_kapita_rp']].copy()


# 05 Penggabungan Indikator Makro dan Ekspor
---
## Integrasi Fitur Ekonomi
---
### Pembentukan Dataset Makro Bersih
---
Menggabungkan data PDRB, pengeluaran, kepadatan, dan luas wilayah, lalu mengekspor ke format CSV.


In [ ]:
kepadatan_dict = {
    'kulon_progo': {'luas_km2': 586.27, 'kepadatan_jiwa_km2': 764.2},
    'bantul': {'luas_km2': 508.85, 'kepadatan_jiwa_km2': 1954.8},
    'gunungkidul': {'luas_km2': 1485.36, 'kepadatan_jiwa_km2': 521.1},
    'sleman': {'luas_km2': 574.82, 'kepadatan_jiwa_km2': 2147.6},
    'kota_yogyakarta': {'luas_km2': 32.50, 'kepadatan_jiwa_km2': 11517.4}
}
df_kepadatan = pd.DataFrame.from_dict(kepadatan_dict, orient='index').reset_index().rename(columns={'index': 'wilayah'})
df_ekonomi = df_pdrb_clean.merge(df_pengeluaran_clean, on='wilayah').merge(df_kepadatan, on='wilayah')
df_ekonomi['pdrb_per_kapita_ribu_rp'] = pd.to_numeric(df_ekonomi['pdrb_per_kapita_ribu_rp'].astype(str).str.replace('.', '').str.replace(',', '.'), errors='coerce')
df_ekonomi['pengeluaran_per_kapita_rp'] = pd.to_numeric(df_ekonomi['pengeluaran_per_kapita_rp'].astype(str).str.replace('.', '').str.replace(',', '.'), errors='coerce')
df_ekonomi


In [ ]:
output_bps_path = OUTPUT_DIR / 'dataset2_bps_makro_cleaned.csv'
df_ekonomi.to_csv(output_bps_path, index=False)
print('Dataset BPS Makro bersih berhasil disimpan di:', output_bps_path)
